In [ ]:
!pip -q install -U pip
!pip -q install "langchain<0.4" "langchain-community<0.4.2" "langchain-core<0.4"
!pip -q install langchain-groq langchain-huggingface "ragas>=0.1.21,<0.3" datasets python-dotenv sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph-sdk 0.4.3 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.


In [ ]:
import json
from pathlib import Path
from google.colab import userdata
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

rows = json.loads(Path("/content/ragas_live_qwen.json").read_text(encoding="utf-8"))
print(len(rows), [r["question"][:16] for r in rows])

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    api_key=userdata.get("GROQ_API_KEY"),
)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

dataset = Dataset.from_dict({
    "question": [r["question"] for r in rows],
    "answer": [r["answer"] for r in rows],
    "contexts": [r["contexts"] for r in rows],
})

6 ['楽天の2025年度の連結Non-', '楽天はAIをどのように活用してい', '日本政府は2030年に対日直接投', '楽天モバイルと楽天エコシステムの', 'GX需要創出のために企業はどのよ', '通商白書で述べられている日本の貿']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings,
)
print(result)
result.to_pandas()

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[11]: RateLimitError(Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.6-27b` in organization `org_01kx9pvjjkexkv0t5rkmhrzser` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1827. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[8]: RateLimitError(Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.6-27b` in organization `org_01kx9pvjjkexkv0t5rkmhrzser` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1391. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tok

{'faithfulness': nan, 'answer_relevancy': nan}


,user_input,retrieved_contexts,response,faithfulness,answer_relevancy
0,楽天の2025年度の連結Non-GAAP営業利益はいくらでしたか？,[の価値創造ストーリー\n財務担当役員メッセージ\nQ 2025年度の業績と課題認識について...,"・1,063億円",NaN,NaN
1,楽天はAIをどのように活用していますか？,[して進めていくことが、私の大切な役割だと考えています。特に最近は、AIの進化が非常に目覚ま...,・マーケティング効率、オペレーション効率、クライアント効率を20%改善する「トリプル20」目...,NaN,NaN
2,日本政府は2030年に対日直接投資残高をいくらに引き上げる目標を掲げていますか？,[在感が増している。また、地域別では最大の欧州は変わらず、アジアが2年ぶりに北米を上回り2位...,・120兆円,NaN,NaN
3,楽天モバイルと楽天エコシステムの関係について説明してください。,[ア、ネットワーク品質に関してユーザーからのレポートがあったエリアへ集中的に資源を投下してい...,・楽天モバイルは楽天エコシステムの中核インフラおよび強力な起点として機能している\n・契約者...,NaN,NaN
4,GX需要創出のために企業はどのような取り組みをすべきですか？,[ＧＸ需要創出に向けた研究会\n中間とりまとめ（参考資料）\n1\nＧＸ需要創出に向けた研究...,・GX製品・サービスを積極的に調達する\n・国の目標を上回る野心的な目標にコミットする\n・...,NaN,NaN
5,通商白書で述べられている日本の貿易上の主な課題は何ですか？,[購入法に基づく基本方針における「印刷」に係る判断の基準にしたがい、印刷用の紙へ\nリサイク...,・財・サービス収支が赤字であること\n・過度な経常収支の不均衡が持続可能ではないこと\n・貿...,NaN,NaN


In [ ]:
import time
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    max_tokens=512,
    api_key=userdata.get("GROQ_API_KEY"),
)

rows_out = []
for i, r in enumerate(rows):
    one = Dataset.from_dict({
        "question": [r["question"]],
        "answer": [r["answer"]],
        "contexts": [r["contexts"]],
    })
    rec = {"i": i, "question": r["question"][:20], "faithfulness": None, "answer_relevancy": None}
    for metric in (faithfulness, answer_relevancy):
        name = metric.name
        try:
            result = evaluate(one, metrics=[metric], llm=llm, embeddings=embeddings)
            rec[name] = float(result[name])
            print(i, name, rec[name])
        except Exception as e:
            rec[name] = f"ERR:{type(e).__name__}:{str(e)[:180]}"
            print(i, name, rec[name])
        time.sleep(20)
    rows_out.append(rec)

pd.DataFrame(rows_out)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


0 faithfulness ERR:TypeError:float() argument must be a string or a real number, not 'list'


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


...

1 faithfulness ERR:TypeError:float() argument must be a string or a real number, not 'list'


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


1 answer_relevancy ERR:TypeError:float() argument must be a string or a real number, not 'list'


KeyboardInterrupt: 

In [ ]:
import time
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=800,
    api_key=userdata.get("GROQ_API_KEY"),
)

def score_of(result, name):
    v = result[name]
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, list) and v and isinstance(v[0], (int, float)):
        return float(v[0])
    return None

out = []
for i, r in enumerate(rows):
    one = Dataset.from_dict({
        "question": [r["question"]],
        "answer": [r["answer"]],
        "contexts": [r["contexts"]],
    })
    rec = {"i": i, "q": r["question"][:18], "faithfulness": None, "answer_relevancy": None}
    for metric in (faithfulness, answer_relevancy):
        try:
            result = evaluate(one, metrics=[metric], llm=llm, embeddings=embeddings)
            rec[metric.name] = score_of(result, metric.name)
            print(i, metric.name, rec[metric.name])
        except Exception as e:
            rec[metric.name] = f"ERR:{type(e).__name__}:{str(e)[:160]}"
            print(i, metric.name, rec[metric.name])
        time.sleep(20)
    out.append(rec)

pd.DataFrame(out)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

0 faithfulness 1.0


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

0 answer_relevancy 0.393734046568232


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

5 faithfulness 1.0


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

5 answer_relevancy 0.6228817219142931


,i,q,faithfulness,answer_relevancy
0,0,楽天の2025年度の連結Non-GA,1.0,0.393734
1,1,楽天はAIをどのように活用しています,NaN,0.668019
2,2,日本政府は2030年に対日直接投資残,1.0,0.522850
3,3,楽天モバイルと楽天エコシステムの関係,NaN,0.847125
4,4,GX需要創出のために企業はどのような,NaN,0.685055
5,5,通商白書で述べられている日本の貿易上,1.0,0.622882


In [ ]:
import time
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=1000,
    api_key=userdata.get("GROQ_API_KEY"),
)

for i in (1, 3, 4):
    r = rows[i]
    one = Dataset.from_dict({
        "question": [r["question"]],
        "answer": [r["answer"]],
        "contexts": [r["contexts"]],
    })
    try:
        result = evaluate(one, metrics=[faithfulness], llm=llm, embeddings=embeddings)
        v = result["faithfulness"]
        print(i, float(v[0] if isinstance(v, list) else v))
    except Exception as e:
        print(i, type(e).__name__, str(e)[:200])
    time.sleep(25)

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


1 nan


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


3 nan


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

4 1.0
